In [1]:
import datetime
import json
import requests
import sys
import time
import numpy as np
import os
import geopandas as gp
import pandas as pd
import osmium
import keyring

In [2]:
# User set the main directory using keyring.  For example:
#    import keyring
#    keyring.set_password('msp', 'vmt_reduction_dir', 'path to the desired directory')
# This will be saved on your local machine

teams_dir = keyring.get_password('msp', 'vmt_reduction_dir')

output_dir = teams_dir + '/2-Re-Routing Analysis/OSM/output'
osm_dir = teams_dir + '/2-Re-Routing Analysis/OSM'
osm_fname = 'analysis-area.osm.pbf'

# Process OSM Data

In [3]:
# Get the complete file directory
osm_file = os.path.join(osm_dir, osm_fname)

## Get Nodes 

In [4]:
# Define a node handler to extract the location of all the nodes in OSM file
class NodesHandler(osmium.SimpleHandler):
    def __init__(self):
        osmium.SimpleHandler.__init__(self)
        self.num_nodes = 0
        self.street_nodes = []
        
    def node(self, n):
        row = { "node_id": n.id, "lat": n.location.lat, "lon": n.location.lon}
        self.street_nodes.append(row)
        self.num_nodes += 1
        
node_handler = NodesHandler()

# Extract all the nodes from the osm file
node_handler.apply_file(osm_file)

print(f"num_nodes: {node_handler.num_nodes}")
osm_nodes = pd.DataFrame(node_handler.street_nodes)

num_nodes: 11585640


In [5]:
# Define a node handler to extract the location of all the signal nodes in OSM file
class TrafficSignalHandler(osmium.SimpleHandler):
    def __init__(self):
        osmium.SimpleHandler.__init__(self)
        self.num_signals = 0
        self.street_signals = []
        
    def node(self, n):
        if n.tags.get('highway') == 'traffic_signals':
            row = { "node_id": n.id, "lat": n.location.lat, "lon": n.location.lon}
            self.street_signals.append(row)
            self.num_signals += 1

signal_handler = TrafficSignalHandler()
signal_handler.apply_file(osm_file)

# show number of signals
print(f"num_signals: {signal_handler.num_signals}")

osm_nodes_signals = pd.DataFrame(signal_handler.street_signals)
osm_nodes_signals['signal'] = 'yes'

num_signals: 5967


## Ways 

In [6]:
# Define a way handler to extract all the highway ways in OSM file
import shapely.wkb as wkblib
class WaysHandler(osmium.SimpleHandler):
    def __init__(self):
        osmium.SimpleHandler.__init__(self)
        self.num_ways = 0
        self.street_ways = []
        self.street_nodes = {}
        self.wkbfab = osmium.geom.WKBFactory()
        
    def way(self, w):
        if w.tags.get("highway") is not None:
            try:
                wkb = self.wkbfab.create_linestring(w)
                geo = wkblib.loads(wkb, hex=True)
            except:
                return
            row = { "way_id": w.id, "highway": w.tags.get('highway'), "name": w.tags.get('name'), "oneway":w.tags.get('oneway'), "geo": geo}

            self.street_ways.append(row)
            self.num_ways += 1
            
            nodes = [n.ref for n in w.nodes]
            self.street_nodes[w.id] = nodes

In [7]:
way_handler = WaysHandler()
way_handler.apply_file(osm_file, locations=True, idx='flex_mem')

# show number of ways in OSM file
print(f"num_ways: {way_handler.num_ways}")

street_ways_gdf = gp.GeoDataFrame(way_handler.street_ways)
street_ways_gdf = street_ways_gdf.set_geometry("geo")
street_ways_gdf = street_ways_gdf.set_crs('epsg:4326')

num_ways: 474592


In [8]:
# Select tertiary and higher level streets for Streetlight data extraction
hstreets = ['motorway', 'motorway_link', 'trunk', 'trunk_link', 'primary', 'primary_link',
 'tertiary', 'tertiary_link', 'secondary', 'secondary_link']
street_ways_gdf_highlevel = street_ways_gdf[street_ways_gdf['highway'].isin(hstreets)].reset_index()

### Intermediate nodes in ways 

In [9]:
# Get all the constituent nodes and their sequence along a way
nodes_in_ways = {'lat':{}, 'lon':{}, 'way_id':{}, 'way_seqid':{}}  
nodeids_in_ways = {'node_id': {}, 'way_id':{}} #not all node ids are included for ways at study area border
rn_id = 0
rec_id = 0
for idx_w in range(len(street_ways_gdf_highlevel)):
    w_id = street_ways_gdf_highlevel.loc[idx_w, 'way_id']
    for nodeid in way_handler.street_nodes[w_id]:
        nodeids_in_ways['way_id'][rn_id] = w_id
        nodeids_in_ways['node_id'][rn_id] = nodeid
        rn_id +=1
    for idx_n, crds in enumerate(list(street_ways_gdf_highlevel.loc[idx_w, 'geo'].coords)):
        nodes_in_ways['lat'][rec_id] =crds[1]
        nodes_in_ways['lon'][rec_id] = crds[0]
        nodes_in_ways['way_id'][rec_id] = w_id
        nodes_in_ways['way_seqid'][rec_id] = idx_n
        rec_id += 1

In [10]:
# Convert dict to dataframe and match with nodes in osm_nodes dataframe
selected_nodes_in_ways_df = pd.DataFrame(nodeids_in_ways)
selected_nodes_in_ways_df = selected_nodes_in_ways_df.merge(osm_nodes, on=['node_id'])
print('number of selected nodes in ways ', len(selected_nodes_in_ways_df))

number of selected nodes in ways  523671


In [11]:
# Convert dict to dataframe
nodes_in_ways_df = pd.DataFrame(nodes_in_ways)
print('number of nodes in ways ', len(nodes_in_ways_df))
nodes_in_ways_df = nodes_in_ways_df.merge(selected_nodes_in_ways_df, on=['lat', 'lon', 'way_id'], how='left')
print('number of nodes without id: ', len(nodes_in_ways_df[pd.isnull(nodes_in_ways_df['node_id'])]))

number of nodes in ways  523671
number of nodes without id:  0


In [12]:
# Get the count of ways that contain each node
nodes_cnts = nodes_in_ways_df.groupby(['node_id', 'lat', 'lon']).size().reset_index(name='counts')

In [13]:
intermediate_nodes_cnts = nodes_in_ways_df.merge(nodes_cnts, on=['node_id', 'lat', 'lon'], how='left')
print('number of nodes without id: ', len(intermediate_nodes_cnts[pd.isnull(intermediate_nodes_cnts['counts'])]))

number of nodes without id:  0


In [14]:
intermediate_nodes_cnts = intermediate_nodes_cnts.merge(osm_nodes_signals, on=['node_id', 'lat', 'lon'], how='left')
print('number of signal nodes : ', len(intermediate_nodes_cnts[intermediate_nodes_cnts['signal']=='yes']))

number of signal nodes :  16834


In [15]:
# Determine nodes to split osm ways: at least two ways cross or signals
intermediate_nodes_cnts['split'] = np.where((intermediate_nodes_cnts['counts']>1) | (intermediate_nodes_cnts['signal']=='yes'), 1, 0)

In [16]:
# Convert to GeoDataFrame
intermediate_nodes_gdf = gp.GeoDataFrame(intermediate_nodes_cnts, geometry=gp.points_from_xy(intermediate_nodes_cnts['lon'], intermediate_nodes_cnts['lat']))
intermediate_nodes_gdf = intermediate_nodes_gdf.set_crs('epsg:4326')

### Splitting ways at intersections or signal nodes

In [17]:
from shapely.geometry import LineString
wayid_list = intermediate_nodes_cnts.way_id.unique().tolist()
ways_splits = {'linkid':{}, 'fr_lat':{}, 'fr_lon':{}, 'fr_node':{}, 'to_lat':{}, 'to_lon':{}, 'to_node':{},
              'way_id':{}, 'way_seqid':{}, 'geometry':{}}
lid = 0
for widx, wid in enumerate(wayid_list):
    way_nodes = intermediate_nodes_cnts[intermediate_nodes_cnts['way_id']==wid].reset_index()
    check_list = way_nodes.split.tolist()
    check_list[0] = 0 # not split at first node
    check_list[-1] = 0 # not split at last node
    split_indices = [i for i, x in enumerate(check_list) if x == 1]
    seqid = 0
    for start, end in zip([0, *split_indices], [*split_indices, len(check_list)]):
        ways_splits['linkid'][lid] = lid
        ways_splits['fr_lat'][lid] = way_nodes.loc[start,'lat']
        ways_splits['fr_lon'][lid] = way_nodes.loc[start,'lon']
        ways_splits['fr_node'][lid] = way_nodes.loc[start,'node_id']
        if end == len(check_list):
            ways_splits['to_lat'][lid] = way_nodes.loc[end-1,'lat']
            ways_splits['to_lon'][lid] = way_nodes.loc[end-1,'lon']
            ways_splits['to_node'][lid] = way_nodes.loc[end-1,'node_id']
        else:
            ways_splits['to_lat'][lid] = way_nodes.loc[end,'lat']
            ways_splits['to_lon'][lid] = way_nodes.loc[end,'lon']
            ways_splits['to_node'][lid] = way_nodes.loc[end,'node_id']
        ways_splits['way_id'][lid] = wid
        ways_splits['way_seqid'][lid] = seqid
        geometry = [xy for xy in zip(way_nodes.loc[start:end]['lon'], way_nodes.loc[start:end]['lat'])]
        ways_splits['geometry'][lid] = LineString(geometry)
        seqid += 1
        lid += 1
    
    if widx%15000==0:
        print('Processed ways_splits %s percent'% (round(100*widx/len(wayid_list),1)))

Processed ways_splits 0.0 percent
Processed ways_splits 28.0 percent
Processed ways_splits 56.0 percent
Processed ways_splits 84.1 percent


In [18]:
#Convert dict to geodataframe
way_splits_links = gp.GeoDataFrame(ways_splits)
way_splits_links = way_splits_links.set_crs('epsg:4326')

In [19]:
b_nodes = way_splits_links[['fr_node', 'fr_lat', 'fr_lon']]
b_nodes.columns = ['node_id', 'lat', 'lon']

e_nodes = way_splits_links[['to_node', 'to_lat', 'to_lon']]
e_nodes.columns = ['node_id', 'lat', 'lon']

way_splits_endnodes = pd.concat([b_nodes, e_nodes], ignore_index=True)

# Get unique node id and number of links that intersect at the node
way_splits_endnodes_cnt=way_splits_endnodes.groupby(['node_id', 'lat', 'lon']).size().reset_index(name='counts')

In [20]:
# attach the node counts back to the splitted link dataframe
cur_cols = way_splits_endnodes_cnt.columns
way_splits_endnodes_cnt.columns = ['fr_node', 'fr_lat', 'fr_lon', 'fr_counts']
way_splits_links = way_splits_links.merge(way_splits_endnodes_cnt, on=['fr_node', 'fr_lat', 'fr_lon'], how='left')

way_splits_endnodes_cnt.columns = ['to_node', 'to_lat', 'to_lon', 'to_counts']
way_splits_links = way_splits_links.merge(way_splits_endnodes_cnt, on=['to_node', 'to_lat', 'to_lon'], how='left')

way_splits_endnodes_cnt.columns = cur_cols

### Get OSM ways endnodes 

In [21]:
# Define function to get coordinates of begin and end points of each link
def latlong(g):
    return g.coords.xy[1][0], g.coords.xy[0][0], g.coords.xy[1][-1], g.coords.xy[0][-1]

In [22]:
street_ways_gdf_highlevel['fr_lat'], street_ways_gdf_highlevel['fr_lon'], street_ways_gdf_highlevel['to_lat'], street_ways_gdf_highlevel['to_lon'] = zip(*street_ways_gdf_highlevel['geo'].map(latlong))

In [23]:
way_b_nodes = street_ways_gdf_highlevel[['fr_lat', 'fr_lon']]
way_b_nodes.columns = ['lat', 'lon']

way_e_nodes = street_ways_gdf_highlevel[['to_lat', 'to_lon']]
way_e_nodes.columns = ['lat', 'lon']

way_endnodes = pd.concat([way_b_nodes, way_e_nodes], ignore_index=True)

# Get unique node id
way_endnodes_cnt=way_endnodes.groupby(['lat', 'lon']).size().reset_index(name='counts')

In [24]:
way_endnodes_cnt = way_endnodes_cnt.merge(nodes_cnts[['lat', 'lon', 'node_id']], on=['lat', 'lon'], how='left')
print('number of nodes without id: ', len(way_endnodes_cnt[pd.isnull(way_endnodes_cnt['node_id'])]))

number of nodes without id:  0


In [25]:
# Generate the the unique node shapefile  
way_endnodes_cnt_gdf = gp.GeoDataFrame(way_endnodes_cnt, geometry=gp.points_from_xy(way_endnodes_cnt.lon, way_endnodes_cnt.lat))
way_endnodes_cnt_gdf = way_endnodes_cnt_gdf.set_crs('epsg:4326')

In [26]:
street_ways_gdf_highlevel['bidir'] = np.where(street_ways_gdf_highlevel['oneway']=='yes', 0, 1)

In [27]:
en_cols = way_endnodes_cnt.columns
way_endnodes_cnt.columns =['fr_lat', 'fr_lon', 'fr_counts', 'fr_node', 'geometry']
ways_highlevel = street_ways_gdf_highlevel.merge(way_endnodes_cnt[['fr_lat', 'fr_lon', 'fr_counts', 'fr_node']], on=['fr_lat', 'fr_lon'], how='left')

way_endnodes_cnt.columns =['to_lat', 'to_lon', 'to_counts', 'to_node', 'geometry']
ways_highlevel = ways_highlevel.merge(way_endnodes_cnt[['to_lat', 'to_lon', 'to_counts', 'to_node']], on=['to_lat', 'to_lon'], how='left')

way_endnodes_cnt.columns = en_cols

In [28]:
# Define a function to check if street name or type of operation changes between two consecutive links
def check_street_name_and_dir(x):
    if x['counts'] == 2:
        nodeid = x['node_id']
        if len(ways_highlevel[ways_highlevel['fr_node']==nodeid]) == 2:
            lidx = ways_highlevel.index[ways_highlevel['fr_node']==nodeid].tolist()
            str_name1 = ways_highlevel.loc[lidx[0], 'name']
            str_name2 = ways_highlevel.loc[lidx[1], 'name']
            dir1 = ways_highlevel.loc[lidx[0], 'bidir']
            dir2 = ways_highlevel.loc[lidx[1], 'bidir']
        elif len(ways_highlevel[ways_highlevel['to_node']==nodeid]) == 2:
            lidx = ways_highlevel.index[ways_highlevel['to_node']==nodeid].tolist()
            str_name1 = ways_highlevel.loc[lidx[0], 'name']
            str_name2 = ways_highlevel.loc[lidx[1], 'name']
            dir1 = ways_highlevel.loc[lidx[0], 'bidir']
            dir2 = ways_highlevel.loc[lidx[1], 'bidir']
        else:
            try:
                str_name1 = ways_highlevel[ways_highlevel['fr_node']==nodeid]['name'].tolist()[0]
                str_name2 = ways_highlevel[ways_highlevel['to_node']==nodeid]['name'].tolist()[0]
                dir1 = ways_highlevel[ways_highlevel['fr_node']==nodeid]['bidir'].tolist()[0]
                dir2 = ways_highlevel[ways_highlevel['to_node']==nodeid]['bidir'].tolist()[0]
            except:
                return 1, 'check'
        if str_name1==str_name2:
            if dir1==dir2:
                return 1, None
            else:
                return 0, 'dir change'
        else:
            return 0, 'name change'
    else:
        return 1, None

In [29]:
way_endnodes_cnt_gdf[['no_change', 'flag']] = way_endnodes_cnt_gdf.apply(lambda x: check_street_name_and_dir(x), axis=1, result_type='expand')

### Merge shorter links between two bounding nodes

In [30]:
# Attach signal info to the splitted links dataframe
intm_cols = osm_nodes_signals.columns
osm_nodes_signals.columns =['fr_node', 'fr_lat', 'fr_lon', 'fr_signal']
way_splits_links_agg = way_splits_links.merge(osm_nodes_signals[['fr_node', 'fr_lat', 'fr_lon', 'fr_signal']], on=['fr_node', 'fr_lat', 'fr_lon'], how='left')

osm_nodes_signals.columns =['to_node', 'to_lat', 'to_lon', 'to_signal']
way_splits_links_agg = way_splits_links_agg.merge(osm_nodes_signals[['to_node', 'to_lat', 'to_lon', 'to_signal']], on=['to_node', 'to_lat', 'to_lon'], how='left')

osm_nodes_signals.columns = intm_cols

In [31]:
# Attach street name or type of operation change info to the splitted links dataframe
intm_cols = way_endnodes_cnt_gdf.columns
way_endnodes_cnt_gdf.columns =['fr_lat', 'fr_lon', 'counts', 'fr_node', 'geometry', 'fr_namedir', 'flag']
way_splits_links_agg = way_splits_links_agg.merge(way_endnodes_cnt_gdf[['fr_node', 'fr_lat', 'fr_lon', 'fr_namedir']], on=['fr_node', 'fr_lat', 'fr_lon'], how='left')

way_endnodes_cnt_gdf.columns =['to_lat', 'to_lon', 'counts', 'to_node', 'geometry', 'to_namedir', 'flag']
way_splits_links_agg = way_splits_links_agg.merge(way_endnodes_cnt_gdf[['to_node', 'to_lat', 'to_lon', 'to_namedir']], on=['to_node', 'to_lat', 'to_lon'], how='left')

way_endnodes_cnt_gdf.columns = intm_cols

In [32]:
# Define the check node where more than 2 links intersect, or it is a signal, or street name changes, or type of operation changes
way_splits_links_agg['fr_check'] = np.where(((way_splits_links_agg['fr_counts']!=2) | (way_splits_links_agg['fr_signal']=='yes') | (way_splits_links_agg['fr_namedir']==0)), 1, 2)
way_splits_links_agg['to_check'] = np.where(((way_splits_links_agg['to_counts']!=2) | (way_splits_links_agg['to_signal']=='yes') | (way_splits_links_agg['to_namedir']==0)), 1, 2)

In [33]:
# Merge shorter links between two bounding nodes into one long segment
start_time = time.time()
# New dataframe to store merged segment attributes
way_splits_links_segs = pd.DataFrame(columns = ['seg_id', 'fr_node', 'fr_lon','fr_lat', 'to_node', 'to_lon','to_lat', 'num_links'])
seg=0
way_splits_links_agg['seg_id']=0
way_splits_links_agg['seqid']=0
for i in range (0,len(way_splits_links_agg)):
    if way_splits_links_agg.loc[i, 'seg_id']==0:
        if way_splits_links_agg.loc[i, 'fr_check']!=2:
            seg_seq=0
            seg+=1
            if way_splits_links_agg.loc[i, 'to_check']!=2:
                way_splits_links_agg.loc[i, 'seg_id']=seg
                way_splits_links_segs.loc[seg-1, 'seg_id']=seg
                way_splits_links_segs.loc[seg-1, 'fr_node']=way_splits_links_agg.loc[i, 'fr_node']
                way_splits_links_segs.loc[seg-1, 'fr_lon']=way_splits_links_agg.loc[i, 'fr_lon']
                way_splits_links_segs.loc[seg-1, 'fr_lat']=way_splits_links_agg.loc[i, 'fr_lat']
                way_splits_links_segs.loc[seg-1, 'to_node']=way_splits_links_agg.loc[i, 'to_node']
                way_splits_links_segs.loc[seg-1, 'to_lon']=way_splits_links_agg.loc[i, 'to_lon']
                way_splits_links_segs.loc[seg-1, 'to_lat']=way_splits_links_agg.loc[i, 'to_lat']
                way_splits_links_segs.loc[seg-1, 'num_links']=1
            else:
                way_splits_links_agg.loc[i, 'seg_id']=seg
                way_splits_links_agg.loc[i, 'seqid']=seg_seq
                way_splits_links_segs.loc[seg-1, 'seg_id']=seg
                way_splits_links_segs.loc[seg-1, 'fr_node']=way_splits_links_agg.loc[i, 'fr_node']
                way_splits_links_segs.loc[seg-1, 'fr_lon']=way_splits_links_agg.loc[i, 'fr_lon']
                way_splits_links_segs.loc[seg-1, 'fr_lat']=way_splits_links_agg.loc[i, 'fr_lat']
                
                preID=way_splits_links_agg.loc[i, 'fr_node']
                curID=way_splits_links_agg.loc[i,'to_node']
                if len(way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist())==0:
                    if len(way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist())==0:
                        # To deal with closed loop
                        idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID)].index.tolist()[0]
                    else:
                        idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist()[0]
                    beginnode_chg=way_splits_links_agg.loc[idx_chg,'fr_node']
                    fr_lon_chg=way_splits_links_agg.loc[idx_chg,'fr_lon']
                    fr_lat_chg=way_splits_links_agg.loc[idx_chg,'fr_lat']
                    fr_check_chg=way_splits_links_agg.loc[idx_chg,'fr_check']

                    way_splits_links_agg.loc[idx_chg,'fr_node']=way_splits_links_agg.loc[idx_chg,'to_node']
                    way_splits_links_agg.loc[idx_chg,'fr_lon']=way_splits_links_agg.loc[idx_chg,'to_lon']
                    way_splits_links_agg.loc[idx_chg,'fr_lat']=way_splits_links_agg.loc[idx_chg,'to_lat']
                    way_splits_links_agg.loc[idx_chg,'fr_check']=way_splits_links_agg.loc[idx_chg,'to_check']   
                    way_splits_links_agg.loc[idx_chg,'to_node']=beginnode_chg
                    way_splits_links_agg.loc[idx_chg,'to_lon']=fr_lon_chg
                    way_splits_links_agg.loc[idx_chg,'to_lat']=fr_lat_chg
                    way_splits_links_agg.loc[idx_chg,'to_check']=fr_check_chg
                    
                idx=way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist()[0]
                if way_splits_links_agg.loc[idx,'fr_check']!=2:
                    print('Not same connecting point ', way_splits_links_agg.loc[i,'linkid'], way_splits_links_agg.loc[idx,'linkid'])
                else:
                    nextID=way_splits_links_agg.loc[idx,'to_node']
                    nextCnt=way_splits_links_agg.loc[idx,'to_check']
                    while nextCnt==2:
                        way_splits_links_agg.loc[idx, 'seg_id']=seg
                        seg_seq+=1
                        way_splits_links_agg.loc[idx, 'seqid']=seg_seq
                        
                        preID=way_splits_links_agg.loc[idx,'fr_node']
                        curID=nextID
                        if len(way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist())==0:
                            if len(way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist())==0:
                                # To deal with closed loop
                                idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID)].index.tolist()[0]
                            else:
                                idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist()[0]                            
                            beginnode_chg=way_splits_links_agg.loc[idx_chg,'fr_node']
                            fr_lon_chg=way_splits_links_agg.loc[idx_chg,'fr_lon']
                            fr_lat_chg=way_splits_links_agg.loc[idx_chg,'fr_lat']
                            fr_check_chg=way_splits_links_agg.loc[idx_chg,'fr_check']

                            way_splits_links_agg.loc[idx_chg,'fr_node']=way_splits_links_agg.loc[idx_chg,'to_node']
                            way_splits_links_agg.loc[idx_chg,'fr_lon']=way_splits_links_agg.loc[idx_chg,'to_lon']
                            way_splits_links_agg.loc[idx_chg,'fr_lat']=way_splits_links_agg.loc[idx_chg,'to_lat']
                            way_splits_links_agg.loc[idx_chg,'fr_check']=way_splits_links_agg.loc[idx_chg,'to_check']   
                            way_splits_links_agg.loc[idx_chg,'to_node']=beginnode_chg
                            way_splits_links_agg.loc[idx_chg,'to_lon']=fr_lon_chg
                            way_splits_links_agg.loc[idx_chg,'to_lat']=fr_lat_chg
                            way_splits_links_agg.loc[idx_chg,'to_check']=fr_check_chg
                            
                        idx=way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist()[0]
                        if way_splits_links_agg.loc[idx,'fr_check']!=2:
                            print('Not same connecting point ', way_splits_links_agg.loc[i,'linkid'], way_splits_links_agg.loc[idx,'linkid'])
                            break
                        nextID=way_splits_links_agg.loc[idx,'to_node']
                        nextCnt=way_splits_links_agg.loc[idx,'to_check']
                    way_splits_links_agg.loc[idx, 'seg_id']=seg
                    seg_seq+=1
                    way_splits_links_agg.loc[idx, 'seqid']=seg_seq
                    way_splits_links_segs.loc[seg-1, 'to_node']=way_splits_links_agg.loc[idx, 'to_node']
                    way_splits_links_segs.loc[seg-1, 'to_lon']=way_splits_links_agg.loc[idx, 'to_lon']
                    way_splits_links_segs.loc[seg-1, 'to_lat']=way_splits_links_agg.loc[idx, 'to_lat']
                    way_splits_links_segs.loc[seg-1, 'num_links']=seg_seq+1
    if i%20000==0:
        print('Link merging processed %s percent'% (round(100*i/len(way_splits_links_agg),3)))
        
while len(way_splits_links_agg.linkid[(way_splits_links_agg.seg_id == 0) & (way_splits_links_agg.to_check != 2)].index.tolist())>0:
    idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.seg_id == 0) & (way_splits_links_agg.to_check != 2)].index.tolist()[0]
    beginnode_chg=way_splits_links_agg.loc[idx_chg,'fr_node']
    fr_lon_chg=way_splits_links_agg.loc[idx_chg,'fr_lon']
    fr_lat_chg=way_splits_links_agg.loc[idx_chg,'fr_lat']
    fr_check_chg=way_splits_links_agg.loc[idx_chg,'fr_check']

    way_splits_links_agg.loc[idx_chg,'fr_node']=way_splits_links_agg.loc[idx_chg,'to_node']
    way_splits_links_agg.loc[idx_chg,'fr_lon']=way_splits_links_agg.loc[idx_chg,'to_lon']
    way_splits_links_agg.loc[idx_chg,'fr_lat']=way_splits_links_agg.loc[idx_chg,'to_lat']
    way_splits_links_agg.loc[idx_chg,'fr_check']=way_splits_links_agg.loc[idx_chg,'to_check']   
    way_splits_links_agg.loc[idx_chg,'to_node']=beginnode_chg
    way_splits_links_agg.loc[idx_chg,'to_lon']=fr_lon_chg
    way_splits_links_agg.loc[idx_chg,'to_lat']=fr_lat_chg
    way_splits_links_agg.loc[idx_chg,'to_check']=fr_check_chg
    
    seg_seq=0
    seg+=1
    way_splits_links_agg.loc[idx_chg, 'seg_id']=seg
    way_splits_links_agg.loc[idx_chg, 'seqid']=seg_seq
    way_splits_links_segs.loc[seg-1, 'seg_id']=seg
    way_splits_links_segs.loc[seg-1, 'fr_node']=way_splits_links_agg.loc[idx_chg, 'fr_node']
    way_splits_links_segs.loc[seg-1, 'fr_lon']=way_splits_links_agg.loc[idx_chg, 'fr_lon']
    way_splits_links_segs.loc[seg-1, 'fr_lat']=way_splits_links_agg.loc[idx_chg, 'fr_lat']
    
    preID=way_splits_links_agg.loc[idx_chg,'fr_node']
    curID=way_splits_links_agg.loc[idx_chg,'to_node']
    if len(way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist())==0:
        if len(way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist())==0:
            # To deal with closed loop
            idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID)].index.tolist()[0]
        else:
            idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist()[0]
        beginnode_chg=way_splits_links_agg.loc[idx_chg,'fr_node']
        fr_lon_chg=way_splits_links_agg.loc[idx_chg,'fr_lon']
        fr_lat_chg=way_splits_links_agg.loc[idx_chg,'fr_lat']
        fr_check_chg=way_splits_links_agg.loc[idx_chg,'fr_check']

        way_splits_links_agg.loc[idx_chg,'fr_node']=way_splits_links_agg.loc[idx_chg,'to_node']
        way_splits_links_agg.loc[idx_chg,'fr_lon']=way_splits_links_agg.loc[idx_chg,'to_lon']
        way_splits_links_agg.loc[idx_chg,'fr_lat']=way_splits_links_agg.loc[idx_chg,'to_lat']
        way_splits_links_agg.loc[idx_chg,'fr_check']=way_splits_links_agg.loc[idx_chg,'to_check']   
        way_splits_links_agg.loc[idx_chg,'to_node']=beginnode_chg
        way_splits_links_agg.loc[idx_chg,'to_lon']=fr_lon_chg
        way_splits_links_agg.loc[idx_chg,'to_lat']=fr_lat_chg
        way_splits_links_agg.loc[idx_chg,'to_check']=fr_check_chg

    idx=way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist()[0]
    if way_splits_links_agg.loc[idx,'fr_check']!=2:
        print('Not same connecting point ', way_splits_links_agg.loc[i,'linkid'], way_splits_links_agg.loc[idx,'linkid'])
    else:
        nextID=way_splits_links_agg.loc[idx,'to_node']
        nextCnt=way_splits_links_agg.loc[idx,'to_check']
        while nextCnt==2:
            way_splits_links_agg.loc[idx, 'seg_id']=seg
            seg_seq+=1
            way_splits_links_agg.loc[idx, 'seqid']=seg_seq
            
            preID=way_splits_links_agg.loc[idx,'fr_node']
            curID=nextID
            if len(way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist())==0:
                if len(way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist())==0:
                    # To deal with closed loop
                    idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID)].index.tolist()[0]
                else:
                    idx_chg=way_splits_links_agg.linkid[(way_splits_links_agg.to_node == curID) & (way_splits_links_agg.fr_node != preID)].index.tolist()[0]  
                beginnode_chg=way_splits_links_agg.loc[idx_chg,'fr_node']
                fr_lon_chg=way_splits_links_agg.loc[idx_chg,'fr_lon']
                fr_lat_chg=way_splits_links_agg.loc[idx_chg,'fr_lat']
                fr_check_chg=way_splits_links_agg.loc[idx_chg,'fr_check']

                way_splits_links_agg.loc[idx_chg,'fr_node']=way_splits_links_agg.loc[idx_chg,'to_node']
                way_splits_links_agg.loc[idx_chg,'fr_lon']=way_splits_links_agg.loc[idx_chg,'to_lon']
                way_splits_links_agg.loc[idx_chg,'fr_lat']=way_splits_links_agg.loc[idx_chg,'to_lat']
                way_splits_links_agg.loc[idx_chg,'fr_check']=way_splits_links_agg.loc[idx_chg,'to_check']   
                way_splits_links_agg.loc[idx_chg,'to_node']=beginnode_chg
                way_splits_links_agg.loc[idx_chg,'to_lon']=fr_lon_chg
                way_splits_links_agg.loc[idx_chg,'to_lat']=fr_lat_chg
                way_splits_links_agg.loc[idx_chg,'to_check']=fr_check_chg
            idx=way_splits_links_agg.linkid[(way_splits_links_agg.fr_node == curID)].index.tolist()[0]
            if way_splits_links_agg.loc[idx,'fr_check']!=2:
                print('Not same connecting point ', way_splits_links_agg.loc[i,'linkid'], way_splits_links_agg.loc[idx,'linkid'])
                break
            nextID=way_splits_links_agg.loc[idx,'to_node']
            nextCnt=way_splits_links_agg.loc[idx,'to_check']

        way_splits_links_agg.loc[idx, 'seg_id']=seg
        seg_seq+=1
        way_splits_links_agg.loc[idx, 'seqid']=seg_seq
        way_splits_links_segs.loc[seg-1, 'to_node']=way_splits_links_agg.loc[idx, 'to_node']
        way_splits_links_segs.loc[seg-1, 'to_lon']=way_splits_links_agg.loc[idx, 'to_lon']
        way_splits_links_segs.loc[seg-1, 'to_lat']=way_splits_links_agg.loc[idx, 'to_lat']
        way_splits_links_segs.loc[seg-1, 'num_links']=seg_seq+1

print("--- %s seconds ---" % (time.time() - start_time))

Link merging processed 0.0 percent
Link merging processed 29.947 percent
Link merging processed 59.895 percent
Link merging processed 89.842 percent
--- 207.4100432395935 seconds ---


In [34]:
way_splits_links_agg.rename(columns={'seqid':'seg_seqid'}, inplace=True)
way_splits_links_merged = way_splits_links.merge(way_splits_links_agg[['linkid', 'seg_id', 'seg_seqid']], on='linkid', how='left')
print('number of links not aggregated: ', len(way_splits_links_merged[pd.isnull(way_splits_links_merged['seg_id'])]))

number of links not aggregated:  0


In [35]:
way_splits_links_merged = way_splits_links_merged.merge(street_ways_gdf_highlevel[['way_id', 'highway', 'name', 'oneway', 'bidir']], on='way_id', how='left')

In [36]:
seg_b_nodes = way_splits_links_segs[['fr_node', 'fr_lon','fr_lat']]
seg_b_nodes.columns = ['node_id', 'lon', 'lat']
seg_e_nodes = way_splits_links_segs[['to_node', 'to_lon', 'to_lat']]
seg_e_nodes.columns = ['node_id', 'lon', 'lat']

seg_endnodes = pd.concat([seg_b_nodes, seg_e_nodes], ignore_index=True)

# Attach node ids
seg_endnodes_cnt=seg_endnodes.groupby(['node_id', 'lon', 'lat']).size().reset_index(name='counts')
seg_endnodes_cnt_gdf = gp.GeoDataFrame(seg_endnodes_cnt, geometry=gp.points_from_xy(seg_endnodes_cnt.lon, seg_endnodes_cnt.lat))
seg_endnodes_cnt_gdf = seg_endnodes_cnt_gdf.set_crs('epsg:4326')

In [37]:
seg_endnodes_cnt_gdf = seg_endnodes_cnt_gdf.merge(osm_nodes_signals[['node_id', 'lat', 'lon', 'signal']], on=['node_id', 'lat', 'lon'], how='left')
seg_endnodes_cnt_gdf = seg_endnodes_cnt_gdf.merge(way_endnodes_cnt_gdf[['node_id', 'lat', 'lon', 'no_change', 'flag']], on=['node_id', 'lat', 'lon'], how='left')

In [38]:
from shapely import geometry, ops
osm_segs = {'seg_id':{}, 'highway':{}, 'st_name':{}, 'bidir':{}, 'geometry':{}}
way_splits_links_sorted = way_splits_links_merged.sort_values(by=['seg_id', 'seg_seqid'], ignore_index=True)
osm_segids = way_splits_links_sorted.seg_id.unique().tolist()
for sid in osm_segids:
    lidx = way_splits_links_sorted.index[way_splits_links_sorted['seg_id']==sid].tolist()
    lidx_sel = lidx[0]
    if len(lidx)>1:
        multi_line = geometry.MultiLineString(way_splits_links_sorted[way_splits_links_sorted['seg_id']==sid].geometry.tolist())
        merged_line = ops.linemerge(multi_line)
        if merged_line.is_empty:
            osm_segs['geometry'][sid] = way_splits_links_sorted.loc[lidx_sel, 'geometry']
        else:
            osm_segs['geometry'][sid] = merged_line
    else:
        osm_segs['geometry'][sid] = way_splits_links_sorted.loc[lidx_sel, 'geometry']
    osm_segs['seg_id'][sid] = sid
    osm_segs['highway'][sid] = way_splits_links_sorted.loc[lidx_sel, 'highway']
    osm_segs['st_name'][sid] = way_splits_links_sorted.loc[lidx_sel, 'name']
    osm_segs['bidir'][sid] = way_splits_links_sorted.loc[lidx_sel, 'bidir']
    
#Convert dict to geodataframe
osm_segs_gdf = gp.GeoDataFrame(osm_segs)
osm_segs_gdf = osm_segs_gdf.set_crs('epsg:4326')

In [39]:
# Define function to create LineStrings for the opposite direction
def reverse_geom(geom):
    def _reverse(x, y, z=None):
        if z:
            return x[::-1], y[::-1], z[::-1]
        return x[::-1], y[::-1]

    return ops.transform(_reverse, geom)

In [40]:
osm_segs_gdf['dir_name'] = 'forward'

In [41]:
osm_segs_gdf = osm_segs_gdf.merge(way_splits_links_segs, on='seg_id', how='left')

In [42]:
osm_segs_gdf.insert(0, 'unqid', range(0, len(osm_segs_gdf)))

In [43]:
osm_segs_oppdir = osm_segs_gdf[osm_segs_gdf['bidir']==1].reset_index()
osm_segs_oppdir['geometry'] = osm_segs_oppdir.apply(lambda x: reverse_geom(x['geometry']), axis=1)
del osm_segs_oppdir['index']
del osm_segs_oppdir['unqid']
osm_segs_oppdir.insert(0, 'unqid', range(osm_segs_gdf.unqid.max()+1, osm_segs_gdf.unqid.max() + 1 + len(osm_segs_oppdir)))
osm_segs_oppdir['dir_name'] = 'reverse'

osm_segs_oppdir['temp_node']= osm_segs_oppdir['fr_node']
osm_segs_oppdir['temp_lat']= osm_segs_oppdir['fr_lat']
osm_segs_oppdir['temp_lon']= osm_segs_oppdir['fr_lon']

osm_segs_oppdir['fr_node']= osm_segs_oppdir['to_node']
osm_segs_oppdir['fr_lat']= osm_segs_oppdir['to_lat']
osm_segs_oppdir['fr_lon']= osm_segs_oppdir['to_lon']

osm_segs_oppdir['to_node']= osm_segs_oppdir['temp_node']
osm_segs_oppdir['to_lat']= osm_segs_oppdir['temp_lat']
osm_segs_oppdir['to_lon']= osm_segs_oppdir['temp_lon']

del osm_segs_oppdir['temp_node']
del osm_segs_oppdir['temp_lat']
del osm_segs_oppdir['temp_lon']

C:\Users\xzh263\Anaconda3\lib\site-packages\pandas\core\dtypes\cast.py:122: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)


In [44]:
osm_segs_bidir_combined = pd.concat([osm_segs_gdf, osm_segs_oppdir], ignore_index=True)

In [45]:
nodes_in_segs = {'lat':{}, 'lon':{}, 'seg_id':{}, 'seg_seqid':{}}
rec_id = 0
for idx_w in range(len(osm_segs_gdf)):
    for idx_n, crds in enumerate(list(osm_segs_gdf.loc[idx_w, 'geometry'].coords)):
        nodes_in_segs['lat'][rec_id] =crds[1]
        nodes_in_segs['lon'][rec_id] = crds[0]
        nodes_in_segs['seg_id'][rec_id] = osm_segs_gdf.loc[idx_w, 'seg_id']
        nodes_in_segs['seg_seqid'][rec_id] = idx_n
        rec_id += 1
        
nodes_in_segs_df = pd.DataFrame(nodes_in_segs)

In [46]:
intermediate_nodes_unique = intermediate_nodes_cnts.drop_duplicates(subset=['node_id', 'lat', 'lon'], ignore_index=True)[['node_id', 'lat', 'lon']]

In [47]:
nodes_in_segs_df = nodes_in_segs_df.merge(intermediate_nodes_unique, on=['lat', 'lon'], how='left')

## Save Intermediate Files if Needed

In [ ]:
# Convert signal nodes to GeoDataFrame and save as shapefile
osm_nodes_signals_gdf = gp.GeoDataFrame(osm_nodes_signals, geometry=gp.points_from_xy(osm_nodes_signals['lon'], osm_nodes_signals['lat']))
osm_nodes_signals_gdf = osm_nodes_signals_gdf.set_crs('epsg:4326')
osm_nodes_signals_gdf.to_file(os.path.join(osm_dir, 'osm_street_nodes_signals.shp'))

In [ ]:
# Save ways with highway as tag and tertiary and above street ways
street_ways_gdf.to_file(os.path.join(osm_dir, 'osm_street_ways.shp'))
street_ways_gdf_highlevel.to_file(os.path.join(osm_dir, 'osm_street_ways_highlevel.shp'))

In [ ]:
# Save all intermediate nodes in ways
intermediate_nodes_gdf.to_file(os.path.join(osm_dir, 'osm_street_ways_intermediate_nodes.shp'))
# Save endnodes of ways
way_endnodes_cnt_gdf.to_file(os.path.join(osm_dir, 'osm_street_ways_endnodes.shp'))

In [ ]:
# Save splitted links
way_splits_links.to_file(os.path.join(osm_dir, 'osm_street_ways_splitted_links.shp'))

# Save splitted links with segment ids that are used for streetlight data extraction
way_splits_links_merged.to_file(os.path.join(osm_dir, 'osm_street_ways_splitted_links_segseq.shp'))

In [ ]:
# Save merged segments
osm_segs_gdf[osm_segs_gdf['geometry'].is_valid].to_file(os.path.join(osm_dir, 'osm_street_ways_splitted_links_merged_segs.shp'))
# Segment endnodes
seg_endnodes_cnt_gdf.to_file(os.path.join(osm_dir, 'osm_street_ways_splitted_links_aggrseg_endnodes.shp'))

In [ ]:
# Directional segments
scols = ['seg_id', 'fr_lat', 'fr_lon', 'to_lat', 'to_lon', 'highway', 'st_name', 'dir_name', 'geometry']
osm_segs_bidir_combined[scols].to_file(os.path.join(osm_dir, 'osm_street_ways_splitted_links_merged_segs_dirs.shp'))

# Create and Upload Streetlight Zone Set

## Prepare zone files 

In [53]:
import math
def get_bearing(lat1, long1, lat2, long2):
    dLon = (long2 - long1)
    x = math.cos(math.radians(lat2)) * math.sin(math.radians(dLon))
    y = math.cos(math.radians(lat1)) * math.sin(math.radians(lat2)) - math.sin(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.cos(math.radians(dLon))
    angle = np.arctan2(x,y)
    brng = round(np.degrees(angle),3)
    if brng < 0:
        return brng + 360
    if brng ==0:
        return 0.1  # Streetlight API couldn't handle a direction of 0, changing it to 0.1 solves the issue
    return brng

In [82]:
# Streetlight required attributes
osm_segs_bidir_combined['id'] = osm_segs_bidir_combined['unqid']
osm_segs_bidir_combined['name'] = osm_segs_bidir_combined['seg_id'].astype(str) + '_' + osm_segs_bidir_combined['dir_name']
osm_segs_bidir_combined['is_pass'] = 1
osm_segs_bidir_combined['is_bidi'] = osm_segs_bidir_combined['bidir']

In [83]:
osm_segs_bidir_combined['direction'] = osm_segs_bidir_combined.apply(lambda x: get_bearing(x['fr_lat'], x['fr_lon'], x['to_lat'], x['to_lon']), axis=1)

In [84]:
osm_segs_bidir_combined['gate_size'] = np.where(osm_segs_bidir_combined['highway'].isin(['motorway', 'motorway_link', 'trunk', 'trunk_link', 'primary','primary_link']), 
                                     40,
                                  np.where(osm_segs_bidir_combined['highway'].isin(['secondary', 'secondary_link']), 30, 20)) # meters, streetlight default

In [85]:
osm_segs_bidir_combined['is_bidi'] = 0

In [87]:
scols = ['id', 'seg_id', 'fr_lat', 'fr_lon', 'to_lat', 'to_lon', 'highway', 'st_name', 'name', 'dir_name', 'direction', 'gate_size', 'geometry']
osm_segs_bidir_combined[scols].to_file(os.path.join(osm_dir, 'osm_street_ways_splitted_links_merged_segs_dirs.shp'))

In [91]:
osm_segs_bidir_combined['is_bidi'] = 0

In [94]:
sub_size = 993
chunk_size = len(osm_segs_bidir_combined)//sub_size + (len(osm_segs_bidir_combined) % sub_size > 0)
print(chunk_size)
zone_prefix = 'mc_osm_merged_'
outcols = ['id', 'name', 'direction', 'is_pass', 'is_bidi', 'gate_size', 'geometry']
for sub_id in range(chunk_size):
    osm_segs_bidir_combined[sub_size * sub_id :  sub_size * (sub_id + 1)][outcols].to_file(os.path.join(osm_dir, zone_prefix + str(sub_id) + '.geojson'), driver="GeoJSON")

63


## Upload zone files 

In [49]:
from config import streetlight_api_key, streetlight_login_email

In [96]:
for sub_id in range(0,chunk_size):
    with open(os.path.join(osm_dir, zone_prefix + str(sub_id) + '.geojson'), 'r') as f:
        osm_segs_json = json.load(f)

    upload_zone_name = zone_prefix + str(sub_id)

    # Create a Zone Set.
    ZONE_SET_REQUEST = {
        "insight_login_email": streetlight_login_email,
        "zone_set_name": upload_zone_name,
        "geom_type": "line",
        "zones": osm_segs_json
    }

    resp = requests.post(
        'https://insight.streetlightdata.com/api/v2/zone_sets',
        headers = {'content-type': 'application/json', 'x-stl-key': streetlight_api_key},
        data = json.dumps(ZONE_SET_REQUEST))

    if (resp.status_code == 201):
        print("Successfully created Zone Set " + str(sub_id))
    else:
        print("Error creating Zone Set " + str(sub_id))

Successfully created Zone Set 0
Successfully created Zone Set 1
Successfully created Zone Set 2
Successfully created Zone Set 3
Successfully created Zone Set 4
Successfully created Zone Set 5
Successfully created Zone Set 6
Successfully created Zone Set 7
Successfully created Zone Set 8
Successfully created Zone Set 9
Successfully created Zone Set 10
Successfully created Zone Set 11
Successfully created Zone Set 12
Successfully created Zone Set 13
Successfully created Zone Set 14
Successfully created Zone Set 15
Successfully created Zone Set 16
Successfully created Zone Set 17
Successfully created Zone Set 18
Successfully created Zone Set 19
Successfully created Zone Set 20
Successfully created Zone Set 21
Successfully created Zone Set 22
Successfully created Zone Set 23
Successfully created Zone Set 24
Successfully created Zone Set 25
Successfully created Zone Set 26
Successfully created Zone Set 27
Successfully created Zone Set 28
Successfully created Zone Set 29
Successfully created

# Create Analysis

In [97]:
hstr = ''
for hour in range(6,20):
    if hour<10:
        hr = '0' + str(hour)
    else:
        hr = str(hour)
    hstr = hstr + 'Hour' + hr + '|' + hr + hr +', '
hstr

'Hour06|0606, Hour07|0707, Hour08|0808, Hour09|0909, Hour10|1010, Hour11|1111, Hour12|1212, Hour13|1313, Hour14|1414, Hour15|1515, Hour16|1616, Hour17|1717, Hour18|1818, Hour19|1919, '

In [99]:
# Create a Segment Analysis.
for sub_id in range(0,chunk_size):
    
    upload_zone_name = zone_prefix + str(sub_id)
    analysis_name = 'vmt_project_sa_' + str(sub_id)
    
    CREATE_ANALYSIS_REQUEST = {
        "insight_login_email": streetlight_login_email,
        "analysis_name": analysis_name,
        "analysis_type": "Segment_Analysis",
        "travel_mode_type": "All_Vehicles",
        "description": "",
        "oz_sets": [{"name":upload_zone_name}],
        #"dz_sets": [{"name":upload_zone_name}],
        "date_ranges": [{'start_date': "01/01/2019", 'end_date': "12/31/2019"}],
        "day_types": "Weekday|15, Saturday|66, Sunday|77",
        "day_parts": "Early AM|0005, Hour06|0606, Hour07|0707, Hour08|0808, Hour09|0909, Hour10|1010, Hour11|1111, Hour12|1212, Hour13|1313, Hour14|1414, Hour15|1515, Hour16|1616, Hour17|1717, Hour18|1818, Hour19|1919, Late PM|2023"
    }

    resp = requests.post(
        'https://insight.streetlightdata.com/api/v2/analyses',
        headers = {'content-type': 'application/json', 'x-stl-key': streetlight_api_key},
        data = json.dumps(CREATE_ANALYSIS_REQUEST))

    if (resp.status_code == 201):
        print("Successfully created analysis for Zone Set " + str(sub_id))
        time.sleep(5)
    else:
        print("Error creating analysis for Zone Set " + str(sub_id))

Successfully created analysis for Zone Set 0
Successfully created analysis for Zone Set 1
Successfully created analysis for Zone Set 2
Successfully created analysis for Zone Set 3
Successfully created analysis for Zone Set 4
Successfully created analysis for Zone Set 5
Successfully created analysis for Zone Set 6
Successfully created analysis for Zone Set 7
Successfully created analysis for Zone Set 8
Successfully created analysis for Zone Set 9
Successfully created analysis for Zone Set 10
Successfully created analysis for Zone Set 11
Successfully created analysis for Zone Set 12
Successfully created analysis for Zone Set 13
Successfully created analysis for Zone Set 14
Successfully created analysis for Zone Set 15
Successfully created analysis for Zone Set 16
Successfully created analysis for Zone Set 17
Successfully created analysis for Zone Set 18
Successfully created analysis for Zone Set 19
Successfully created analysis for Zone Set 20
Successfully created analysis for Zone Set 2

In [100]:
# Create an AADT analysis.
aadt_year = 2019
for sub_id in range(0,chunk_size):
    
    upload_zone_name = zone_prefix + str(sub_id)
    analysis_name = 'vmt_project_aadt_' + str(sub_id)
    
    CREATE_ANALYSIS_REQUEST = {
        "insight_login_email": streetlight_login_email,
        "analysis_name": analysis_name,
        "analysis_type": "AADT",
        "travel_mode_type": "All_Vehicles",
        "description": "",
        "oz_sets": [{"name":upload_zone_name}],
        "aadt_year": aadt_year
    }

    resp = requests.post(
        'https://insight.streetlightdata.com/api/v2/analyses',
        headers = {'content-type': 'application/json', 'x-stl-key': streetlight_api_key},
        data = json.dumps(CREATE_ANALYSIS_REQUEST))


    if (resp.status_code == 201):
        print("Successfully created analysis for Zone Set " + str(sub_id))
        time.sleep(5)
    else:
        print("Error creating analysis for Zone Set " + str(sub_id))

Successfully created analysis for Zone Set 0
Successfully created analysis for Zone Set 1
Successfully created analysis for Zone Set 2
Successfully created analysis for Zone Set 3
Successfully created analysis for Zone Set 4
Successfully created analysis for Zone Set 5
Successfully created analysis for Zone Set 6
Successfully created analysis for Zone Set 7
Successfully created analysis for Zone Set 8
Successfully created analysis for Zone Set 9
Successfully created analysis for Zone Set 10
Successfully created analysis for Zone Set 11
Successfully created analysis for Zone Set 12
Successfully created analysis for Zone Set 13
Successfully created analysis for Zone Set 14
Successfully created analysis for Zone Set 15
Successfully created analysis for Zone Set 16
Successfully created analysis for Zone Set 17
Successfully created analysis for Zone Set 18
Successfully created analysis for Zone Set 19
Successfully created analysis for Zone Set 20
Successfully created analysis for Zone Set 2

# Check Analysis Status - Caution: Queries can take a long time! It is advised to check analysis status on Streetlight InSignt Platform

In [50]:
def print_response(response):
    print("response code: {}".format(response.status_code))
    print("response body: {}".format(response.content))

In [52]:
# Check the processing status of the Analysis.
analysis_name = 'vmt_project_sa_0'

CHECK_STATUS_REQUEST = {
    "analyses":[{"name": analysis_name}]
}

while True:
    resp = requests.post(
        'https://insight.streetlightdata.com/api/v2/analyses/status',
        headers = {'content-type': 'application/json', 'x-stl-key': streetlight_api_key},
        data = json.dumps(CHECK_STATUS_REQUEST))

    print_response(resp)

    if (resp.status_code != 200):
        print("Error checking Analysis Status.")
        sys.exit(1)

    json_result = json.loads(resp.text)
    analysis_status = json_result["analyses"][0]["status"]

    if (analysis_status in ["Available", "Data_Available", "Data Available"]):
        print("Analysis is Available!")
        break
    elif (analysis_status == "Processing"):
        print("Analysis is processing. Trying again after 5 minutes...")
        time.sleep(300)
    else:
        print("Error running Analysis.")
        sys.exit(1)

response code: 200
response body: b'{"analyses":[{"metrics":["sa_all","sample_size"],"name":"vmt_project_sa_0","status":"Data Available","uuid":"e253758c-def1-4c2b-ad32-1ab4b3c9bfaa"}],"status":"success"}\n'
Analysis is Available!


# Get Analysis Results

In [53]:
import io
from config import streetlight_api_key, streetlight_login_email

In [54]:
# Get the segment analysis results.
chunk_size = 63
all_speeds = pd.DataFrame()
for sub_id in range(chunk_size):
    analysis_name = 'vmt_project_sa_' + str(sub_id)
    resp = requests.get(
        'https://insight.streetlightdata.com/api/v2/analyses/download/name/{}/sa_all'.format(analysis_name),
        headers = {'x-stl-key': streetlight_api_key})

    if (resp.status_code == 200):
        # save the results as dataframe
        urlData = resp.content
        spdData = pd.read_csv(io.StringIO(urlData.decode('utf-8')))
        all_speeds = pd.concat([all_speeds, spdData])
        print("Success fetching data for chunk %s with %s records." % (sub_id, len(spdData)))
    else:
        print("Error fetching query results for chunk %s." % sub_id)
        sys.exit(1)

Success fetching data for chunk 0 with 66365 records.
Success fetching data for chunk 1 with 65611 records.
Success fetching data for chunk 2 with 62988 records.
Success fetching data for chunk 3 with 64095 records.
Success fetching data for chunk 4 with 63409 records.
Success fetching data for chunk 5 with 63534 records.
Success fetching data for chunk 6 with 65289 records.
Success fetching data for chunk 7 with 64275 records.
Success fetching data for chunk 8 with 61611 records.
Success fetching data for chunk 9 with 66538 records.
Success fetching data for chunk 10 with 67023 records.
Success fetching data for chunk 11 with 66271 records.
Success fetching data for chunk 12 with 66234 records.
Success fetching data for chunk 13 with 65629 records.
Success fetching data for chunk 14 with 65992 records.
Success fetching data for chunk 15 with 66185 records.
Success fetching data for chunk 16 with 65023 records.
Success fetching data for chunk 17 with 66503 records.
Success fetching dat

In [55]:
# Get the query results.
all_aadt = pd.DataFrame()
output_metric  = 'estimated_aadt'

for sub_id in range(chunk_size):
    analysis_name = 'vmt_project_aadt_' + str(sub_id)
    resp = requests.get(
        'https://insight.streetlightdata.com/api/v2/analyses/download/name/{}/{}'.format(analysis_name, output_metric),
        headers = {'x-stl-key': streetlight_api_key})

    if (resp.status_code == 200):
        # save the results as dataframe
        urlData = resp.content
        volData = pd.read_csv(io.StringIO(urlData.decode('utf-8')))
        all_aadt = pd.concat([all_aadt, volData])
        print("Success fetching aadt data for chunk %s with %s records." % (sub_id, len(volData)))
        time.sleep(5)
    else:
        print("Error fetching query results for chunk %s." % sub_id)
        sys.exit(1)

Success fetching aadt data for chunk 0 with 989 records.
Success fetching aadt data for chunk 1 with 981 records.
Success fetching aadt data for chunk 2 with 959 records.
Success fetching aadt data for chunk 3 with 974 records.
Success fetching aadt data for chunk 4 with 982 records.
Success fetching aadt data for chunk 5 with 955 records.
Success fetching aadt data for chunk 6 with 982 records.
Success fetching aadt data for chunk 7 with 983 records.
Success fetching aadt data for chunk 8 with 948 records.
Success fetching aadt data for chunk 9 with 986 records.
Success fetching aadt data for chunk 10 with 992 records.
Success fetching aadt data for chunk 11 with 985 records.
Success fetching aadt data for chunk 12 with 981 records.
Success fetching aadt data for chunk 13 with 972 records.
Success fetching aadt data for chunk 14 with 979 records.
Success fetching aadt data for chunk 15 with 982 records.
Success fetching aadt data for chunk 16 with 975 records.
Success fetching aadt da

In [60]:
# Save streetlight speed and aadt data
all_speeds.to_csv(os.path.join(output_dir, 'streetlight_speeds_extracted.csv'), index=False)
all_aadt.to_csv(os.path.join(output_dir, 'streetlight_aadt_extracted.csv'), index=False)

# Prepare Data for Routing Network

In [48]:
use_cols =['Data Periods', 'Mode of Travel', 'Zone ID', 'Zone Name', 'Line Zone Length (Miles)', 'Zone Is Pass-Through',
       'Zone Direction (degrees)', 'Zone is Bi-Direction', 'Day Type', 'Day Part', 'Average Daily Segment Traffic (StL Index)',
       'Avg Segment Speed (mph)', 'Free Flow Speed (mph)']

# Read in saved Streetlight speed data
congestion = pd.read_csv(os.path.join(output_dir, 'streetlight_speeds_extracted.csv'), usecols = use_cols)
congestion.head(1)

,Data Periods,Mode of Travel,Zone ID,Zone Name,Line Zone Length (Miles),Zone Is Pass-Through,Zone Direction (degrees),Zone is Bi-Direction,Day Type,Day Part,Average Daily Segment Traffic (StL Index),Avg Segment Speed (mph),Free Flow Speed (mph)
0,"Jan 01, 2019 - Dec 31, 2019",All Vehicles LBS Plus - StL All Vehicles Index,0,1_forward,0.309,yes,191,no,3: Sunday (Su-Su),14: Hour18 (6pm-7pm),131,23.0,27.652


In [49]:
ffs = congestion[['Zone Name', 'Free Flow Speed (mph)']].drop_duplicates(['Zone Name'], ignore_index=True)

In [50]:
ffs_notna = ffs[~pd.isnull(ffs['Free Flow Speed (mph)'])]

In [51]:
congestion = congestion.sort_values(by=['Zone Name', 'Day Type', 'Day Part'], ignore_index=True)

In [52]:
congestion['day_period'] = congestion['Day Type'] + '_' + congestion['Day Part']

In [53]:
congestion_sel = congestion[congestion['Day Type']!='0: All Days (M-Su)']
congestion_sel = congestion_sel[congestion_sel['Day Part']!='00: All Day (12am-12am)']
congestion_sel = congestion_sel[congestion_sel['Zone Name'].isin(ffs_notna['Zone Name'])]

In [54]:
# transpose the congestion dataframe
congestion_t = congestion_sel.pivot(index='Zone Name', columns='day_period', values='Avg Segment Speed (mph)').reset_index()

In [55]:
p = ['0-6']
for h in range(6,20):
    p.append(str(h) + '-' + str(h+1))
p.append('20-24')
d = ['weedays', 'saturdays', 'sundays']
dp = ['weedays_' + ep for ep in p] + ['saturdays_' + ep for ep in p] + ['sundays' + ep for ep in p]

In [56]:
congestion_t.columns = ['seg_dir'] + dp

In [57]:
congestion_t = congestion_t.merge(ffs_notna, left_on='seg_dir', right_on=['Zone Name'], how='left')
print('null ffs segments', len(congestion_t[pd.isnull(congestion_t['Free Flow Speed (mph)'])]))

null ffs segments 0


In [58]:
fillna_cols = congestion_t.columns[1:-2]
for c in fillna_cols:
    congestion_t[c] = congestion_t[c].fillna(congestion_t['Free Flow Speed (mph)'])

In [59]:
congestion_t['seg_id']= congestion_t["seg_dir"].str.split("_").str[0]
congestion_t['seg_id'] = congestion_t['seg_id'].astype(int)
congestion_t['direction']= congestion_t["seg_dir"].str.split("_").str[1]

In [60]:
n_cols = ['seg_id', 'direction', 'seg_dir', 'weedays_0-6', 'weedays_6-7', 'weedays_7-8', 'weedays_8-9',
       'weedays_9-10', 'weedays_10-11', 'weedays_11-12', 'weedays_12-13',
       'weedays_13-14', 'weedays_14-15', 'weedays_15-16', 'weedays_16-17',
       'weedays_17-18', 'weedays_18-19', 'weedays_19-20', 'weedays_20-24',
       'saturdays_0-6', 'saturdays_6-7', 'saturdays_7-8', 'saturdays_8-9',
       'saturdays_9-10', 'saturdays_10-11', 'saturdays_11-12',
       'saturdays_12-13', 'saturdays_13-14', 'saturdays_14-15',
       'saturdays_15-16', 'saturdays_16-17', 'saturdays_17-18',
       'saturdays_18-19', 'saturdays_19-20', 'saturdays_20-24', 'sundays0-6',
       'sundays6-7', 'sundays7-8', 'sundays8-9', 'sundays9-10', 'sundays10-11',
       'sundays11-12', 'sundays12-13', 'sundays13-14', 'sundays14-15',
       'sundays15-16', 'sundays16-17', 'sundays17-18', 'sundays18-19',
       'sundays19-20', 'sundays20-24']
congestion_t = congestion_t[n_cols]

In [61]:
# Read in saved Streetlight aadt data
aadt = pd.read_csv(os.path.join(output_dir, 'streetlight_aadt_extracted.csv'), usecols = ['Zone Name', 'Day Type', 'Day Part', 'Estimated 2019 AADT'])
aadt.head(1)

,Zone Name,Day Type,Day Part,Estimated 2019 AADT
0,1_forward,0: All Days (M-Su),0: All Day (12am-12am),1284


In [62]:
aadt_ffs = aadt[['Zone Name', 'Estimated 2019 AADT']].merge(ffs_notna, on='Zone Name')
len(aadt_ffs)

59234

In [63]:
aadt_ffs.columns = ['seg_dir', 'AADT', 'FFS']
aadt_ffs['seg_id']= aadt_ffs["seg_dir"].str.split("_").str[0]
aadt_ffs['seg_id'] = aadt_ffs['seg_id'].astype(int)
aadt_ffs['direction']= aadt_ffs["seg_dir"].str.split("_").str[1]

In [65]:
# Get merged segments and constituent ways table
stl_seg_table = way_splits_links_sorted.drop_duplicates(subset=['seg_id', 'way_id']).reset_index()
stl_seg_table = stl_seg_table[['seg_id', 'way_id']].reset_index()
stl_seg_table.columns = ['recid', 'seg_id', 'way_id']

In [67]:
# Only keep segments that have streetlight data
stl_seg_table_update = stl_seg_table[stl_seg_table['seg_id'].isin(congestion_t['seg_id'])]

In [68]:
nodes_in_segs_update = nodes_in_segs_df[nodes_in_segs_df['seg_id'].isin(congestion_t['seg_id'])]

In [69]:
nodes_in_ways_update = nodes_in_ways_df[nodes_in_ways_df['way_id'].isin(stl_seg_table_update['way_id'])]

In [73]:
nodes_table = nodes_in_segs_update.merge(stl_seg_table_update[['seg_id', 'way_id']], on='seg_id')

In [74]:
nodes_table = nodes_table.merge(nodes_in_ways_update, on=['way_id', 'node_id'])

In [218]:
import sqlite3

# Connect to the SQLite database (will create a new file if it doesn't exist)
conn = sqlite3.connect(os.path.join(output_dir, 'streetlight_data.db'))

# Save above dataframes to the SQLite database
aadt_ffs.to_sql('stl_aadt_ffs_2019', conn, if_exists='replace', index=False)
congestion_t.to_sql('stl_congestion_data_2019', conn, if_exists='replace', index=False)
nodes_table.to_sql('stl_nodes_table', conn, if_exists='replace', index=False)
stl_seg_table_update.to_sql('stl_segments_table', conn, if_exists='replace', index=False)
nodes_in_ways_update.to_sql('stl_nodes_in_ways', conn, if_exists='replace', index=False)
nodes_in_segs_update.to_sql('stl_nodes_in_segments', conn, if_exists='replace', index=False)

# Close the database connection
conn.close()